In [1]:
import numpy as np
import pandas as pd
import os

print(os.getcwd())
print(os.listdir())
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets"))
print(os.listdir("/kaggle/input/datasets/jhffmn"))
BASE_PATH = "/kaggle/input/datasets/jhffmn/diabetes-data-set"

/kaggle/working
['__notebook__.ipynb']
['datasets']
['jhffmn']
['diabetes-data-set']


## Introduction

This notebook demonstrates a categorical Naive Bayes classifier built for structured health datasets such as diabetes and heart disease data. The purpose of the notebook is not only to show that the classifier works, but also to explain the logic of the algorithm in a way that supports a later implementation in C, including future parallelization with OpenMP and MPI.

A classifier is a program that learns from labeled examples and then predicts labels for new, unseen data. In this project, each row represents one patient record, each feature is a categorical variable, and the final column is the class label the model is trying to predict. The classifier learns by examining how often different feature values occur within each class. It then uses those counts to estimate probabilities and decide which class is most likely for a new row.

The model used here is Naive Bayes. The word “Bayes” refers to the idea of estimating the probability of a class given the observed feature values. The word “naive” refers to the simplifying assumption that features are conditionally independent once the class is known. While that assumption is not perfectly true in most real datasets, Naive Bayes often performs well in practice and is especially attractive here because it is simple, interpretable, and based primarily on counting.

This project is also designed with software engineering goals in mind. The Python implementation is being used as a prototype to validate the algorithm, test it on multiple datasets, and clearly separate the logic into functions that can later be translated into C. That means the code is written in a procedural style and avoids unnecessary abstractions. The goal is to make each major step of the algorithm easy to understand, easy to test, and easy to map into lower-level code.

Another major goal of the project is to prepare for parallel execution. Naive Bayes is a good fit for parallel programming because training is largely a counting problem over rows, and prediction is performed independently for each row. In a shared-memory model such as OpenMP, loops over rows can be divided across threads as long as each thread maintains local counts before merging them. In a distributed-memory model such as MPI, rows can be divided across processes, local counts can be computed independently, and then the counts can be reduced into one final global model. Cross-validation also provides another level of parallelism because each fold can be run independently.

This notebook therefore serves four purposes at once. First, it explains the theory behind the classifier. Second, it shows how that theory is implemented in Python. Third, it highlights how each part would need to be translated into C. Fourth, it identifies which parts of the workflow are naturally parallelizable and which parts are not. The result is both a working machine learning prototype and a design document for a future parallel C implementation.

### Load Metadata

#### What this step is doing

The metadata file defines the structure of the dataset. It specifies:
- The names of all columns (features and target)
- The number of allowed categorical values for each column
- The actual allowed values for each feature and the class label

This information is essential because the classifier must know:
- What each column represents
- What values are valid for each feature
- How to interpret the class labels

By using metadata, the classifier becomes **general-purpose** and can work across different datasets (e.g., diabetes and heart disease) without hardcoding assumptions.

---

#### How this is implemented in Python

The metadata file is read as a raw table using Pandas with no header.

- Row 0 → column names  
- Row 1 → number of allowed values per column  
- Rows 2+ → the actual categorical values  

The function then:
1. Extracts column names
2. Extracts the number of allowed values per column
3. Builds a list of allowed values for each column
4. Splits features and target (last column is the class)
5. Creates mappings from categorical values → integer indices

These mappings allow us to convert raw categorical values into array indices for efficient computation.

---

#### How this would be implemented in C

In C, this would require:
- Reading the file line by line (e.g., using `fgets`)
- Parsing values manually (e.g., using `strtok` or `sscanf`)
- Storing:
  - Column names in arrays of strings
  - Allowed values in 2D arrays
- Creating lookup tables to map categorical values to indices

Instead of Python dictionaries, C would likely use:
- Arrays (if values are small and contiguous), or
- Custom lookup logic

All structures would be stored in `struct` types.

---

#### Parallelization considerations

This step is **not performance-critical** and does not benefit much from parallelization.

- It is executed once at the beginning
- The workload is small compared to training and prediction

Therefore:
- It is typically kept **serial**
- No OpenMP or MPI parallelization is needed here

Parallelization efforts should instead focus on:
- Training (count aggregation)
- Prediction (row-wise classification)
- Cross-validation (independent folds)

In [2]:
def load_metadata(meta_path):
    raw = pd.read_csv(meta_path, header=None)

    column_names = raw.iloc[0].astype(str).tolist()
    num_values = raw.iloc[1].astype(int).tolist()

    allowed_values = []
    for col_idx, count in enumerate(num_values):
        column_values = []
        for row_idx in range(2, 2 + count):
            column_values.append(int(raw.iat[row_idx, col_idx]))
        allowed_values.append(column_values)

    feature_names = column_names[:-1]
    target_name = column_names[-1]

    feature_allowed_values = allowed_values[:-1]
    class_values = allowed_values[-1]

    feature_value_to_index = []
    for values in feature_allowed_values:
        feature_value_to_index.append({value: idx for idx, value in enumerate(values)})

    class_to_index = {value: idx for idx, value in enumerate(class_values)}

    return (
        column_names,
        feature_names,
        target_name,
        feature_allowed_values,
        feature_value_to_index,
        class_values,
        class_to_index,
    )

### Load Labeled and Unlabeled Data

#### What this step is doing (Theory)

This step loads the datasets and ensures they match the structure defined in the metadata.  
- The labeled dataset includes both features and the target (class label).  
- The unlabeled dataset includes only features.  

Validating column order and names ensures the model interprets the data correctly.

---

#### How this is implemented in Python

- Read the CSV file using Pandas  
- Verify that the column names match the expected metadata  
- Convert all values to integers for efficient computation  

---

#### How this would be implemented in C

In C, this would involve:
- Reading the CSV file line by line  
- Parsing values manually  
- Storing data in arrays (e.g., 2D arrays for features)  
- Verifying column order using predefined metadata  

---

#### Parallelization considerations

This step is **not parallelized**:
- It is performed once at the start  
- The cost is small compared to training and prediction  

In [3]:
def load_labeled_data(csv_path, column_names):
    df = pd.read_csv(csv_path)
    if list(df.columns) != column_names:
        raise ValueError("Labeled CSV columns do not match metadata.")
    return df.astype(int)


def load_unlabeled_data(csv_path, feature_names):
    df = pd.read_csv(csv_path)
    if list(df.columns) != feature_names:
        raise ValueError("Unlabeled CSV columns do not match metadata.")
    return df.astype(int)

### Initialize Count Tables

#### What this step is doing (Theory)

This step initializes the data structures used to store frequency counts during training.

Naive Bayes learns by counting:
- How many times each class appears
- How many times each feature value appears within each class

These counts are the foundation for computing probabilities later.

---

#### How this is implemented in Python

- `class_counts`: an array of size `num_classes` to track how often each class occurs  
- `feature_counts`: a list of 2D arrays  
  - Each array corresponds to one feature  
  - Shape: `(num_classes, number_of_feature_values)`  
  - Tracks how often each feature value appears within each class  
- `total_rows`: keeps track of how many rows have been processed  

All arrays are initialized to zero.

---

#### How this would be implemented in C

In C, this would involve:
- Allocating arrays using `malloc`
- `class_counts` → 1D array of size `num_classes`
- `feature_counts` → array of pointers to 2D arrays  
- Initializing all values to zero (e.g., using loops or `calloc`)  
- Maintaining an integer counter for total rows  

---

#### Parallelization considerations

This step is **not parallelized**:
- It is a one-time setup operation  
- Each thread/process would create its own local version of these structures during parallel training  

In [4]:
def initialize_counts(num_classes, feature_allowed_values):
    class_counts = np.zeros(num_classes, dtype=np.int64)

    feature_counts = []
    for vals in feature_allowed_values:
        feature_counts.append(np.zeros((num_classes, len(vals)), dtype=np.int64))

    total_rows = 0
    return class_counts, feature_counts, total_rows

### Update Counts from a Single Row

#### What this step is doing (Theory)

This is the core operation of Naive Bayes training.

For each row in the dataset, the algorithm:
- Increments the count for the observed class
- Increments the count for each feature value within that class

In other words, it answers:
- “How often does this class occur?”
- “Given this class, how often does each feature value occur?”

By processing all rows this way, the model builds the frequency tables needed to compute probabilities.

---

#### How this is implemented in Python

- Extract the class label from the last element of the row  
- Convert the class label into an index  
- Increment the corresponding entry in `class_counts`  
- Loop over each feature:
  - Convert the feature value into its index  
  - Increment the corresponding entry in `feature_counts`  

This function updates the count tables **one row at a time**.

---

#### How this would be implemented in C

In C, this would involve:
- Accessing the class value from the row array  
- Using a lookup (or direct indexing) to map values to indices  
- Incrementing values in integer arrays  
- Looping over features using standard `for` loops  

All operations are simple integer updates on arrays.

---

#### Parallelization considerations

This function itself is not parallelized. It operates on a single row and performs simple updates.

Parallelization is applied at a higher level (in the training loop), where multiple rows are processed independently. Each thread or process maintains its own local count tables, and these are combined after processing to form the final global counts.

In [5]:
def update_counts_from_row(row, class_counts, feature_counts, total_rows,
                           feature_value_to_index, class_to_index):
    class_value = int(row[-1])
    class_idx = class_to_index[class_value]

    class_counts[class_idx] += 1
    total_rows += 1

    num_features = len(feature_counts)

    for j in range(num_features):
        value = int(row[j])
        value_idx = feature_value_to_index[j][value]
        feature_counts[j][class_idx, value_idx] += 1

    return total_rows

### Collect Counts from Dataset

#### What this step is doing (Theory)

This step processes the entire training dataset and builds the frequency tables used by Naive Bayes.

For each row in the dataset:
- The class count is updated
- The feature-value counts for that class are updated

By iterating over all rows, the algorithm accumulates:
- Total occurrences of each class
- Total occurrences of each feature value within each class

These counts are later used to compute probabilities.

---

#### How this is implemented in Python

- Initialize count tables using `initialize_counts`  
- Loop over each row in the dataset  
- Call `update_counts_from_row` to update the counts  
- Track the total number of processed rows  

The dataset is processed sequentially, one row at a time.

---

#### How this would be implemented in C

In C, this would involve:
- Iterating over a 2D array of data using a loop  
- Passing each row to a function that updates count arrays  
- Maintaining arrays for class counts and feature counts  
- Tracking the total number of rows processed  

This would be implemented using standard `for` loops and array indexing.

---

#### Parallelization considerations

This is the primary location where parallelization is applied.

- The loop over rows can be divided across threads or processes  
- Each worker maintains its own local count tables  
- After processing, local counts are combined (summed) into global counts  

This approach enables efficient parallel training while avoiding race conditions.

In [6]:
def collect_counts(data_array, feature_value_to_index, class_to_index,
                   num_classes, feature_allowed_values):
    class_counts, feature_counts, total_rows = initialize_counts(
        num_classes, feature_allowed_values
    )

    num_rows = data_array.shape[0]

    for row_idx in range(num_rows):
        row = data_array[row_idx]
        total_rows = update_counts_from_row(
            row,
            class_counts,
            feature_counts,
            total_rows,
            feature_value_to_index,
            class_to_index,
        )

    return class_counts, feature_counts, total_rows

### Convert Counts to Probabilities

#### What this step is doing (Theory)

This step converts the raw frequency counts into probabilities used by the Naive Bayes classifier.

Two types of probabilities are computed:

- **Class priors:**  
  The probability of each class occurring in the dataset  
  $$P(c) = \frac{\text{count}(c) + \alpha}{N + \alpha \cdot K}$$

- **Conditional probabilities:**  
  The probability of each feature value given a class  
  $$P(x_j = v \mid c) = \frac{\text{count}(c, v) + \alpha}{\text{count}(c) + \alpha \cdot V_j}$$

Laplace smoothing (controlled by $\alpha$) is used to prevent zero probabilities.

Log probabilities are also computed to:
- Avoid numerical underflow
- Convert multiplication into addition during prediction

---

#### How this is implemented in Python

- Compute class priors using class counts and total rows  
- Take the logarithm of the class priors  
- For each feature:
  - Compute smoothed conditional probabilities  
  - Store both normal probabilities and log probabilities  
- Store results in lists for later use in prediction  

All computations are performed using NumPy arrays for efficiency.

---

#### How this would be implemented in C

In C, this would involve:
- Iterating over arrays of counts using nested loops  
- Computing probabilities using floating-point arithmetic  
- Storing results in:
  - A 1D array for class priors  
  - 2D arrays for conditional probabilities  
- Computing logarithms using functions like `log()` from `<math.h>`  

All operations would be explicit and performed using array indexing.

---

#### Parallelization considerations

This step is typically **not parallelized**:

- It is performed once after training  
- The computational cost is relatively small compared to counting  

Parallelization efforts are better focused on:
- Counting (training phase)
- Prediction (row-wise classification)

In [7]:
def counts_to_probabilities(class_counts, feature_counts, total_rows, alpha):
    num_classes = len(class_counts)

    # Compute class priors with Laplace smoothing
    class_priors = (class_counts + alpha) / (total_rows + alpha * num_classes)
    log_class_priors = np.log(class_priors)

    conditional_probs = []
    log_conditional_probs = []

    # Compute conditional probabilities for each feature
    for feature_idx in range(len(feature_counts)):
        counts = feature_counts[feature_idx]
        num_values = counts.shape[1]

        numerator = counts + alpha
        denominator = class_counts.reshape(-1, 1) + alpha * num_values

        probs = numerator / denominator

        conditional_probs.append(probs)
        log_conditional_probs.append(np.log(probs))

    return class_priors, log_class_priors, conditional_probs, log_conditional_probs


### Train the Naive Bayes Model

#### What this step is doing (Theory)

This step performs the full training process for the Naive Bayes classifier.

Training consists of two main phases:

1. **Counting phase**  
   The algorithm scans the dataset and counts:
   - How often each class occurs  
   - How often each feature value occurs within each class  

2. **Probability computation phase**  
   The counts are converted into:
   - Class prior probabilities  
   - Conditional probabilities for each feature value given a class  

These probabilities form the final model used for prediction.

---

#### How this is implemented in Python

- Call `collect_counts` to compute all frequency counts  
- Call `counts_to_probabilities` to convert counts into probabilities  
- Return both the counts and the computed probability tables  

The function is structured as a simple pipeline:
data → counts → probabilities → model

---

#### How this would be implemented in C

In C, this would involve:
- Calling a function to accumulate counts using loops and arrays  
- Calling a second function to compute probabilities from those counts  
- Storing results in arrays (or structs) for later use  

The structure would closely mirror the Python version, with explicit function calls and array-based data structures.

---

#### Parallelization considerations

This function is where parallelization is coordinated, but not directly implemented.

- The **counting phase** (`collect_counts`) is the primary target for parallelization  
- The **probability computation phase** is typically left serial  

In a parallel implementation:
- Each thread or process would compute local counts  
- Local counts would be combined into global counts  
- The final probability computation would be performed once on the combined counts

In [8]:
def train(data_array, feature_value_to_index, class_to_index,
          num_classes, feature_allowed_values, alpha=1.0):

    # Step 1: collect frequency counts from the dataset
    class_counts, feature_counts, total_rows = collect_counts(
        data_array,
        feature_value_to_index,
        class_to_index,
        num_classes,
        feature_allowed_values,
    )

    # Step 2: convert counts into probabilities
    (
        class_priors,
        log_class_priors,
        conditional_probs,
        log_conditional_probs,
    ) = counts_to_probabilities(
        class_counts,
        feature_counts,
        total_rows,
        alpha,
    )

    return (
        class_counts,
        feature_counts,
        class_priors,
        log_class_priors,
        conditional_probs,
        log_conditional_probs,
    )

### Pedict a single row

#### What this step is doing (Theory)

This function classifies a single data point using the Naive Bayes model.

The goal is to compute the most likely class \( c \) given the observed feature values \( x = (x_1, x_2, \dots, x_n) \):

$$
\hat{c} = \arg\max_c \; P(c \mid x)
$$

Using Bayes’ rule:

$$
P(c \mid x) \propto P(c) \prod_{j=1}^{n} P(x_j \mid c)
$$

Under the Naive Bayes assumption (conditional independence), this becomes:

$$
P(c \mid x) \propto P(c) \cdot \prod_{j=1}^{n} P(x_j \mid c)
$$

To avoid numerical underflow, we take the logarithm:

$$
\log P(c \mid x) = \log P(c) + \sum_{j=1}^{n} \log P(x_j \mid c)
$$

The classifier computes this log score for each class and selects the class with the highest value.

---

#### How this is implemented in Python

- Initialize `log_scores` with the log prior probabilities  
- Loop over each feature in the row:
  - Convert the feature value to an index  
  - Add the corresponding log conditional probability  
- After processing all features, each entry in `log_scores` contains the score for one class  
- Use `argmax` to find the index of the highest score  
- Return the corresponding class label  

---

#### How this would be implemented in C

In C, this would involve:

- Creating an array of log scores (one per class)  
- Initializing it with log priors  
- Looping over features:
  - Mapping feature values to indices  
  - Adding log probabilities from a 2D array  
- Finding the maximum value using a loop  
- Returning the corresponding class label  

All operations are simple loops and array indexing.

---

#### Parallelization considerations

This function itself is not parallelized because it operates on a single row.

However, prediction is highly parallelizable at a higher level:

- Each row can be classified independently  
- The outer loop over rows (in `predict_dataset`) can be parallelized  

This makes prediction an example of **embarrassingly parallel computation**.

In [9]:
def predict_row(row, log_class_priors, log_conditional_probs,
                feature_value_to_index, class_values):
    log_scores = log_class_priors.copy()

    for j in range(len(row)):
        value = int(row[j])
        value_idx = feature_value_to_index[j][value]
        log_scores += log_conditional_probs[j][:, value_idx]

    best_idx = int(np.argmax(log_scores))
    return class_values[best_idx]

### Predict an Entire Dataset

#### What this step is doing (Theory)

This function applies the trained Naive Bayes classifier to an entire dataset.

Given a set of rows  
$$
X = \{x^{(1)}, x^{(2)}, \dots, x^{(m)}\}
$$

the goal is to compute:

$$
\hat{y}^{(i)} = \arg\max_c \; P(c \mid x^{(i)})
$$

for each row $$x^{(i)}$$

Each row is classified independently using the same model, producing a vector of predicted class labels.

---

#### How this is implemented in Python

- Create an array `predictions` to store the output labels  
- Loop over each row in the dataset  
- Call `predict_row` to classify each row  
- Store the predicted class in the output array  
- Return the full array of predictions  

This function acts as a wrapper that applies the single-row classifier repeatedly.

---

#### How this would be implemented in C

In C, this would involve:

- Allocating an array to store predictions  
- Looping over each row of the dataset  
- Calling a function equivalent to `predict_row`  
- Storing the result in the output array  

All operations would be performed using simple loops and array indexing.

---

#### Parallelization considerations

This function is **highly parallelizable**.

- Each row is processed independently  
- No shared state is modified during prediction  

This makes it an example of **embarrassingly parallel computation**

Parallel approaches:

- **OpenMP**: parallelize the loop over rows  
- **MPI**: distribute subsets of rows across processes  

Prediction is often easier to parallelize than training because no merging step is required.

In [10]:
def predict_dataset(data_array, log_class_priors, log_conditional_probs,
                    feature_value_to_index, class_values):
    predictions = np.zeros(data_array.shape[0], dtype=int)

    for i in range(data_array.shape[0]):
        predictions[i] = predict_row(
            data_array[i],
            log_class_priors,
            log_conditional_probs,
            feature_value_to_index,
            class_values,
        )

    return predictions

### Confusion Matrix (Binary Classification)

#### What this step is doing (Theory)

This function computes the **confusion matrix** for a binary classification problem.

A confusion matrix summarizes how well a classifier performs by comparing:
- The **true labels** \( y_{\text{true}} \)
- The **predicted labels** \( y_{\text{pred}} \)

For binary classification (classes 0 and 1), the confusion matrix has four components:

|                | Predicted 0 | Predicted 1 |
|----------------|-------------|-------------|
| **Actual 0**   | TN          | FP          |
| **Actual 1**   | FN          | TP          |

Where:

- **True Negative (TN)**: Model correctly predicts class 0  
- **False Positive (FP)**: Model predicts 1 but actual is 0  
- **False Negative (FN)**: Model predicts 0 but actual is 1  
- **True Positive (TP)**: Model correctly predicts class 1  

---

#### Why this matters

Accuracy alone can be misleading, especially when classes are imbalanced.

The confusion matrix allows us to understand:
- How often the model makes each type of error  
- Whether the model favors one class over another  
- The trade-off between different types of mistakes  

From these values, we compute important metrics:

- **Accuracy**  
  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$

- **Precision** (How many predicted positives are correct)  
  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

- **Recall** (How many actual positives are found)  
  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

- **F1 Score** (Balance of precision and recall)  
  $$
  F1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
  $$

These metrics provide a more complete picture of model performance.

---

#### How this is implemented in Python

- Initialize counters for TN, FP, FN, TP  
- Loop over all examples  
- Compare true label and predicted label  
- Increment the appropriate counter  
- Return all four values  

The logic is implemented using simple conditional checks.

---

#### How this would be implemented in C

In C, this would involve:

- Declaring integer counters for TN, FP, FN, TP  
- Looping over arrays of true and predicted labels  
- Using `if`/`else` conditions to update counts  
- Returning results (e.g., via pointers or a struct)  

This maps directly to simple loop-based logic.

---

#### Parallelization considerations

This computation is **parallelizable**:

- Each row comparison is independent  
- Counts can be computed in parallel using local counters  
- Local results can be combined by summing TN, FP, FN, TP  

This makes it suitable for both:
- **OpenMP** (parallel loop with reduction)
- **MPI** (compute locally, then reduce across processes)

In [11]:
def confusion_matrix_binary(y_true, y_pred):
    tn = 0
    fp = 0
    fn = 0
    tp = 0

    for i in range(len(y_true)):
        yt = int(y_true[i])
        yp = int(y_pred[i])

        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1

    return tn, fp, fn, tp

### Evaluate Predictions

#### What this step is doing (Theory)

This function evaluates how well the classifier performs by comparing predicted labels to true labels.

It computes several standard metrics derived from the **confusion matrix**:

- **True Positive (TP)**: correctly predicted positive class  
- **True Negative (TN)**: correctly predicted negative class  
- **False Positive (FP)**: predicted positive but actually negative  
- **False Negative (FN)**: predicted negative but actually positive  

From these values, we compute:

- **Accuracy**  
  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$

- **Precision** (How many predicted positives are correct)  
  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

- **Recall** (How many actual positives are correctly identified)  
  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

- **F1 Score** (Balance between precision and recall)  
  $$
  F1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
  $$

These metrics provide a more complete understanding of model performance than accuracy alone.

---

#### How this is implemented in Python

- Compute accuracy using a vectorized comparison  
- Call `confusion_matrix_binary` to get TN, FP, FN, TP  
- Compute precision, recall, and F1 score  
- Handle edge cases (division by zero) safely  
- Return all metrics for reporting  

---

#### How this would be implemented in C

In C, this would involve:

- Looping over arrays of true and predicted labels  
- Computing TN, FP, FN, TP using conditional logic  
- Performing floating-point calculations for metrics  
- Handling division-by-zero cases manually  
- Returning results via variables or a struct  

All operations would use basic loops and arithmetic.

---

#### Parallelization considerations

This step is **parallelizable**:

- Each comparison between true and predicted labels is independent  
- Counts (TN, FP, FN, TP) can be computed in parallel using local variables  
- Final results are obtained by summing across threads or processes  

This can be implemented using:
- **OpenMP** (parallel loop with reduction)
- **MPI** (local computation followed by reduction)

In [12]:
def evaluate_predictions(y_true, y_pred):
    accuracy = np.mean(y_true == y_pred)

    tn, fp, fn, tp = confusion_matrix_binary(y_true, y_pred)

    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0

    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0

    if precision + recall > 0:
        f1 = 2.0 * precision * recall / (precision + recall)
    else:
        f1 = 0.0

    return accuracy, precision, recall, f1, tn, fp, fn, tp

### Cross-Validation (K-Fold)

#### What this step is doing (Theory)

Cross-validation is used to evaluate how well a model generalizes to unseen data.

In **k-fold cross-validation**, the dataset is split into \( k \) equal parts (folds):

$$
X = F_1 \cup F_2 \cup \dots \cup F_k
$$

For each fold \( i \):

- Use \( F_i \) as the **test set**
- Use all other folds as the **training set**

$$
\text{Train on } X \setminus F_i, \quad \text{Test on } F_i
$$

This process is repeated \( k \) times, and the results are averaged.

This provides a more reliable estimate of model performance than a single train/test split.

---

#### How this is implemented in Python

- Shuffle the dataset indices and split them into \( k \) folds  
- For each fold:
  - Select one fold as the test set  
  - Combine the remaining folds into the training set  
  - Train the model  
  - Predict on the test set  
  - Compute evaluation metrics  
- Store results for each fold and compute the mean  

This ensures every data point is used for both training and testing.

---

#### How this would be implemented in C

In C, this would involve:

- Creating an array of shuffled indices  
- Splitting indices into \( k \) groups  
- Looping over folds:
  - Constructing training and test index sets  
  - Building training data arrays  
  - Calling training and prediction functions  
- Accumulating metrics and computing averages  

All operations would be implemented using arrays and loops.

---

#### Parallelization considerations

Cross-validation is **highly parallelizable**.

- Each fold is independent  
- No shared state is required between folds  

Parallel strategies:

- **OpenMP**: parallelize the loop over folds  
- **MPI**: assign each fold to a different process  

Additional parallelism exists within each fold:
- Training (row counting) can be parallelized  
- Prediction (row classification) can be parallelized  

This creates multiple levels of parallelism within the workflow.

In [13]:
def make_k_folds(n_rows, k, seed=42):
    indices = np.arange(n_rows)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    return np.array_split(indices, k)


def cross_validate(X, y, feature_value_to_index, class_to_index,
                   num_classes, feature_allowed_values, class_values,
                   k=5, alpha=1.0, seed=42):
    folds = make_k_folds(len(X), k, seed)

    accuracies = []
    precisions = []
    recalls = []
    f1_scores = []

    # Parallelization could happen here:
    # each fold is independent and can be run separately.
    for fold_idx in range(k):
        test_idx = folds[fold_idx]
        train_idx = np.concatenate(
            [folds[j] for j in range(k) if j != fold_idx]
        )

        X_train = X[train_idx]
        y_train = y[train_idx]
        X_test = X[test_idx]
        y_test = y[test_idx]

        train_data = np.column_stack((X_train, y_train))

        # Parallelization inside train() would happen
        # when counting rows in the training data.
        _, _, _, log_class_priors, _, log_conditional_probs = train(
            train_data,
            feature_value_to_index,
            class_to_index,
            num_classes,
            feature_allowed_values,
            alpha,
        )

        # Parallelization inside predict_dataset() would happen
        # over the rows in X_test.
        y_pred = predict_dataset(
            X_test,
            log_class_priors,
            log_conditional_probs,
            feature_value_to_index,
            class_values,
        )

        accuracy, precision, recall, f1, _, _, _, _ = evaluate_predictions(y_test, y_pred)

        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return (
        float(np.mean(accuracies)),
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(f1_scores)),
    )

### Main Program Workflow

#### What this step is doing (Theory)

This section ties together the entire machine learning pipeline from start to finish.

The workflow follows a standard supervised learning process:

1. Load metadata to understand the structure of the dataset  
2. Load labeled and unlabeled data  
3. Train a Naive Bayes model using the labeled data  
4. Evaluate model performance on known labels  
5. Use cross-validation to estimate generalization performance  
6. Apply the trained model to unlabeled data  
7. Save predictions for later use  

This represents a complete end-to-end classification system.

---

#### How this is implemented in Python

The `main()` function organizes the pipeline into clear stages:

- **Setup**: define file paths and model parameters  
- **Metadata loading**: interpret dataset structure  
- **Data loading**: read labeled and unlabeled datasets  
- **Preprocessing**: convert data into integer arrays  
- **Training**: build the Naive Bayes model  
- **Evaluation**: compute metrics on the training dataset  
- **Cross-validation**: estimate performance across multiple splits  
- **Prediction**: classify unlabeled data  
- **Output**: save predictions to a file  

Each stage is implemented using separate functions, making the program modular and easier to understand.

---

#### How this would be implemented in C

In C, this would involve:

- Reading metadata and datasets using file I/O functions  
- Storing data in arrays or structs  
- Calling functions for:
  - Training (counting + probability computation)  
  - Prediction (row-wise classification)  
  - Evaluation (confusion matrix and metrics)  
- Writing results to an output file  

The structure would mirror the Python version but use explicit memory management and array-based operations.

---

#### Parallelization considerations

Parallelization occurs at multiple levels in this workflow:

- **Training phase**:  
  Row-wise counting can be parallelized across threads or processes  

- **Prediction phase**:  
  Each row can be classified independently  

- **Cross-validation**:  
  Each fold can be executed independently  

This creates multiple opportunities for parallel execution using:
- **OpenMP** (shared memory)
- **MPI** (distributed memory)

The `main()` function itself remains serial, but it orchestrates calls to functions where parallelization is applied.

In [14]:
def main():
    # File paths
    meta_path = BASE_PATH + "/diabetes_meta.csv"
    labeled_path = BASE_PATH + "/diabetes_20000_labeled.csv"
    unlabeled_path = BASE_PATH + "/diabetes_20000_unlabeled.csv"
    output_path = "/kaggle/working/diabetes_predictions.csv"

    # Model settings
    alpha = 1.0
    k = 5

    # Load dataset metadata
    (
        column_names,
        feature_names,
        target_name,
        feature_allowed_values,
        feature_value_to_index,
        class_values,
        class_to_index,
    ) = load_metadata(meta_path)

    # Load labeled and unlabeled data
    labeled_df = load_labeled_data(labeled_path, column_names)
    unlabeled_df = load_unlabeled_data(unlabeled_path, feature_names)

    # Convert DataFrames to integer arrays
    labeled_data = labeled_df.to_numpy(dtype=int)
    unlabeled_data = unlabeled_df.to_numpy(dtype=int)

    # Split labeled data into features and true labels
    X = labeled_data[:, :-1]
    y = labeled_data[:, -1]
    num_classes = len(class_values)

    # Train the model
    # Parallelization inside train() would happen during count collection.
    (
        class_counts,
        feature_counts,
        class_priors,
        log_class_priors,
        conditional_probs,
        log_conditional_probs,
    ) = train(
        labeled_data,
        feature_value_to_index,
        class_to_index,
        num_classes,
        feature_allowed_values,
        alpha,
    )

    # Evaluate predictions on the full labeled dataset
    # Parallelization inside predict_dataset() would happen over rows.
    train_pred = predict_dataset(
        X,
        log_class_priors,
        log_conditional_probs,
        feature_value_to_index,
        class_values,
    )

    accuracy, precision, recall, f1, tn, fp, fn, tp = evaluate_predictions(y, train_pred)

    print("Training Results")
    print("----------------")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    # Run k-fold cross-validation
    # Parallelization could occur across folds.
    mean_acc, mean_prec, mean_rec, mean_f1 = cross_validate(
        X,
        y,
        feature_value_to_index,
        class_to_index,
        num_classes,
        feature_allowed_values,
        class_values,
        k=k,
        alpha=alpha,
        seed=42,
    )

    print("\nCross Validation")
    print("----------------")
    print(f"Mean Accuracy : {mean_acc:.4f}")
    print(f"Mean Precision: {mean_prec:.4f}")
    print(f"Mean Recall   : {mean_rec:.4f}")
    print(f"Mean F1 Score : {mean_f1:.4f}")

    # Predict labels for the unlabeled dataset
    # Parallelization here would also happen over rows.
    unlabeled_pred = predict_dataset(
        unlabeled_data,
        log_class_priors,
        log_conditional_probs,
        feature_value_to_index,
        class_values,
    )

    # Save predictions to a CSV file
    output_df = unlabeled_df.copy()
    output_df["Predicted_" + target_name] = unlabeled_pred
    output_df.to_csv(output_path, index=False)

    print(f"\nPredictions saved to {output_path}")


if __name__ == "__main__":
    main()

Training Results
----------------
Accuracy : 0.8157
Precision: 0.3802
Recall   : 0.5113
F1 Score : 0.4361
TN=14890, FP=2323, FN=1362, TP=1425

Cross Validation
----------------
Mean Accuracy : 0.8153
Mean Precision: 0.3795
Mean Recall   : 0.5106
Mean F1 Score : 0.4351

Predictions saved to /kaggle/working/diabetes_predictions.csv
